In [12]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [13]:
!pip install faiss-cpu -q

import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

print("Creating knowledge base")
kb = []
for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

print("Loading embedding model and creating index")
model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = model.encode(kb, show_progress_bar=True)
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print("Knowledge base successfully created:", len(kb), "documents")

Creating knowledge base
Loading embedding model and creating index


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Knowledge base successfully created: 2000 documents


In [14]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=0)  # device=0 uses GPU if available

row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])]
ans_150 = str(row_150[row_150['answer']])

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [15]:
query_embedding_150 = model.encode([prompt_150])
distances, retrieved_indices = index.search(query_embedding_150, k=10)
retrieved_indices = retrieved_indices[0]

# rank is 1-indexed position of index 150 in the retrieved list
if 150 in retrieved_indices:
    q2_answer = int(np.where(retrieved_indices == 150)[0][0]) + 1
else:
    q2_answer = None  # not in top 10
print("Q2 answer:", q2_answer)
print("Retrieved indices:", retrieved_indices)

Q2 answer: 10
Retrieved indices: [ 663 1701 1269 1532  576  847 1693 1906  168  150]


In [16]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

docs_10 = [kb[i] for i in retrieved_indices]
pairs = [[prompt_150, doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)

# Sort retrieved_indices by ce_scores descending
order = np.argsort(ce_scores)[::-1]
sorted_retrieved_indices = retrieved_indices[order]

q3_answer = int(np.where(sorted_retrieved_indices == 150)[0][0]) + 1
print("Q3 answer:", q3_answer)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Q3 answer: 1


In [17]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

row_42 = train.iloc[42]
prompt_42 = str(row_42['prompt'])

query_embedding_42 = model.encode([prompt_42])
_, retrieved_42 = index.search(query_embedding_42, k=5)
retrieved_42 = retrieved_42[0]

docs_42 = [kb[i] for i in retrieved_42]
concatenated_docs = " ".join(docs_42)

rag_string_42 = f"Context: {concatenated_docs} Question: {prompt_42}"
tokens = tokenizer(rag_string_42, truncation=False)
q4_answer = len(tokens['input_ids'])
print("Q4 answer:", q4_answer)

Q4 answer: 216


In [18]:
true_document_150 = kb[150]
rag_string_150 = f"Context: {true_document_150} Question: {prompt_150}"

result_q5 = zs(rag_string_150, labels_150)
scores_dict_q5 = dict(zip(result_q5['labels'], result_q5['scores']))
q5_answer = round(scores_dict_q5[ans_150], 3)
print("Q5 answer:", q5_answer)

Q5 answer: 0.989


In [19]:
adversarial_doc = kb[999]
adversarial_string = f"Context: {adversarial_doc} Question: {prompt_150}"

result_q6 = zs(adversarial_string, labels_150)
scores_dict_q6 = dict(zip(result_q6['labels'], result_q6['scores']))
q6_answer = round(scores_dict_q6[ans_150], 3)
print("Q6 answer:", q6_answer)

Q6 answer: 0.529


In [20]:
hits = 0
for i in range(100):
    row = train.iloc[i]
    prompt_i = str(row['prompt'])
    correct_answer_text = str(row[row['answer']])

    query_emb = model.encode([prompt_i])
    _, ret_idx = index.search(query_emb, k=5)
    ret_idx = ret_idx[0]

    retrieved_docs = [kb[j] for j in ret_idx]
    if any(correct_answer_text in doc for doc in retrieved_docs):
        hits += 1

q7_answer = round((hits / 100) * 100, 1)
print("Q7 answer:", q7_answer)

Q7 answer: 73.0


In [21]:
def map_at_3(ranked_labels, correct_label):
    if correct_label in ranked_labels[:3]:
        rank = ranked_labels.index(correct_label) + 1
        return 1.0 / rank
    return 0.0

ap_scores = []
for i in range(20):
    row = train.iloc[i]
    prompt_i = str(row['prompt'])
    options = [str(row['A']), str(row['B']), str(row['C']), str(row['D']), str(row['E'])]
    letters = ['A', 'B', 'C', 'D', 'E']
    correct_letter = row['answer']

    # Retrieve top 5
    query_emb = model.encode([prompt_i])
    _, ret_idx = index.search(query_emb, k=5)
    ret_idx = ret_idx[0]
    docs_5 = [kb[j] for j in ret_idx]

    # Rerank, pick best doc
    pairs_i = [[prompt_i, doc] for doc in docs_5]
    scores_i = cross_encoder.predict(pairs_i)
    best_doc = docs_5[int(np.argmax(scores_i))]

    # Augment
    rag_string_i = f"Context: {best_doc} Question: {prompt_i}"

    # Predict
    result_i = zs(rag_string_i, options)

    # Map label text back to letter, sort by score
    text_to_letter = {str(row[l]): l for l in letters}
    ranked_letters = [text_to_letter[label] for label in result_i['labels']]  # already sorted desc by score

    ap = map_at_3(ranked_letters, correct_letter)
    ap_scores.append(ap)

q8_answer = round(np.mean(ap_scores), 3)
print("Q8 answer:", q8_answer)

Q8 answer: 0.975
